<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/04_hypothesis_testing_ab_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04: Hypothesis Testing & A/B Testing

> **Goal:** Learn how to design, run, and interpret A/B tests — one of the most valuable skills for data scientists.

**Scenario:** You're a data scientist at an e-commerce company. The product team wants to test a new checkout button.

**What you'll learn:**
1. The 5-step hypothesis testing framework
2. Statistical power and sample size calculation
3. Running A/B tests (frequentist)
4. Bayesian A/B testing (better for business decisions)
5. Common A/B testing mistakes

**Time:** ~3 hours

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import zt_ind_solve_power
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print('Setup complete!')

## Part 1: The 5-Step Hypothesis Testing Framework

**Business Context:** Our checkout page has a red "Buy Now" button with a 5.2% conversion rate. We want to test a green button. We want to detect a 15% relative improvement (5.2% → 5.98%).

In [ ]:
# STEP 1: State the hypotheses
print('STEP 1: HYPOTHESES')
print('H₀ (Null): p_green = p_red  (No difference in conversion rates)')
print('H₁ (Alternative): p_green ≠ p_red  (Two-tailed test)')
print('Or one-tailed: H₁: p_green > p_red  (We believe green is better)')
print()

# Business parameters
p_control = 0.052          # Current conversion rate (red button)
min_detectable_effect = 0.15  # 15% relative improvement = minimum we care about
p_treatment = p_control * (1 + min_detectable_effect)  # = 0.0598

print(f'Control rate:   {p_control:.3f} ({p_control*100:.1f}%)')
print(f'Minimum detectable effect: {min_detectable_effect:.0%} relative lift')
print(f'Treatment rate: {p_treatment:.4f} ({p_treatment*100:.2f}%)')

In [ ]:
# STEP 2: Choose significance level
alpha = 0.05   # Type I error rate: 5% chance of false positive
power = 0.80   # 1 - Type II error rate: 80% chance of detecting real effect

print(f'STEP 2: SIGNIFICANCE LEVEL')
print(f'α (alpha) = {alpha}  → Accept {alpha*100:.0f}% risk of false positive')
print(f'Power = {power}    → {power*100:.0f}% chance of detecting effect if real')
print()
print('Interpretation:')
print(f'  If H₀ is TRUE: We incorrectly reject it {alpha*100:.0f}% of the time (Type I error)')
print(f'  If H₁ is TRUE: We correctly detect it {power*100:.0f}% of the time')

In [ ]:
# STEP 3: Calculate required sample size
from scipy.stats import norm

def sample_size_for_proportions(p1, p2, alpha=0.05, power=0.80, two_tailed=True):
    """
    Calculate minimum sample size per group for a two-proportion z-test.
    Uses Cohen's h as effect size measure.
    """
    # Cohen's h = effect size for proportions
    h = 2 * np.arcsin(np.sqrt(p2)) - 2 * np.arcsin(np.sqrt(p1))

    # Z-values for alpha and power
    z_alpha = norm.ppf(1 - alpha / (2 if two_tailed else 1))
    z_power = norm.ppf(power)

    n = ((z_alpha + z_power) / h) ** 2
    return int(np.ceil(n)), abs(h)


n_per_group, cohens_h = sample_size_for_proportions(p_control, p_treatment)
total_n = n_per_group * 2

print(f'STEP 3: SAMPLE SIZE CALCULATION')
print(f'Cohen\'s h (effect size): {cohens_h:.4f}')
print(f'Required: {n_per_group:,} users per group')
print(f'Total: {total_n:,} users')
print()

# Business translation
daily_visitors = 5000
days_needed = np.ceil(total_n / daily_visitors)
print(f'At {daily_visitors:,} daily visitors: {days_needed:.0f} days ≈ {days_needed/7:.1f} weeks')

In [ ]:
# STEP 4: Run the experiment (simulate)
print('STEP 4: RUN THE EXPERIMENT')
print(f'Collecting {n_per_group:,} users per group...')

# Simulate experiment results
# TRUE effect: green button converts at 6.1% (better than minimum 5.98%)
true_p_treatment = 0.061

control_visitors = n_per_group
treatment_visitors = n_per_group

control_conversions = np.random.binomial(control_visitors, p_control)
treatment_conversions = np.random.binomial(treatment_visitors, true_p_treatment)

obs_p_control = control_conversions / control_visitors
obs_p_treatment = treatment_conversions / treatment_visitors

print(f'\nResults:')
print(f'  Control   (red):   {control_conversions:,}/{control_visitors:,} = {obs_p_control:.4f} ({obs_p_control*100:.2f}%)')
print(f'  Treatment (green): {treatment_conversions:,}/{treatment_visitors:,} = {obs_p_treatment:.4f} ({obs_p_treatment*100:.2f}%)')
print(f'  Observed lift: {(obs_p_treatment/obs_p_control - 1)*100:.1f}%')

In [ ]:
# STEP 5: Statistical test and interpretation
# Two-proportion z-test
count = np.array([treatment_conversions, control_conversions])
nobs = np.array([treatment_visitors, control_visitors])

z_stat, p_value = proportions_ztest(count, nobs, alternative='larger')  # one-tailed

print('STEP 5: STATISTICAL TEST')
print(f'Test: One-tailed two-proportion z-test')
print(f'H₁: p_green > p_red')
print(f'z-statistic: {z_stat:.4f}')
print(f'p-value: {p_value:.4f}')
print()

# Confidence intervals
ci_control = proportion_confint(control_conversions, control_visitors, alpha=0.05, method='wilson')
ci_treatment = proportion_confint(treatment_conversions, treatment_visitors, alpha=0.05, method='wilson')

print(f'95% CI Control:   [{ci_control[0]:.4f}, {ci_control[1]:.4f}]')
print(f'95% CI Treatment: [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]')
print()

# Decision
if p_value < alpha:
    lift = (obs_p_treatment - obs_p_control) / obs_p_control * 100
    print(f'✅ REJECT H₀ (p={p_value:.4f} < α={alpha})')
    print(f'The green button is statistically significantly better.')
    print(f'Estimated lift: {lift:.1f}%')
else:
    print(f'❌ FAIL TO REJECT H₀ (p={p_value:.4f} ≥ α={alpha})')
    print('No significant difference detected.')

## Part 2: Bayesian A/B Testing (Better for Business)

Frequentist gives you a p-value. Bayesian gives you **P(treatment > control)** — much more actionable.

In [ ]:
def bayesian_ab_test(control_conversions, control_total,
                     treatment_conversions, treatment_total,
                     n_samples=200_000):
    """
    Bayesian A/B test using Beta-Binomial conjugate model.
    Prior: Beta(1,1) = uniform = no prior knowledge
    Posterior: Beta(1 + conversions, 1 + non-conversions)
    """
    # Posterior distributions
    alpha_c = 1 + control_conversions
    beta_c  = 1 + (control_total - control_conversions)
    alpha_t = 1 + treatment_conversions
    beta_t  = 1 + (treatment_total - treatment_conversions)

    # Monte Carlo sampling
    samples_c = np.random.beta(alpha_c, beta_c, n_samples)
    samples_t = np.random.beta(alpha_t, beta_t, n_samples)

    # Key metrics
    prob_t_better = np.mean(samples_t > samples_c)
    lift = (samples_t - samples_c) / samples_c
    lift_mean = np.mean(lift)
    lift_ci = np.percentile(lift, [2.5, 97.5])
    expected_loss = np.mean(np.maximum(0, samples_c - samples_t))

    return {
        'prob_treatment_better': prob_t_better,
        'lift_mean': lift_mean,
        'lift_ci_95': lift_ci,
        'expected_loss': expected_loss,
        'samples': (samples_c, samples_t),
        'posteriors': ((alpha_c, beta_c), (alpha_t, beta_t)),
    }


result = bayesian_ab_test(
    control_conversions, control_visitors,
    treatment_conversions, treatment_visitors
)

print('BAYESIAN A/B TEST RESULTS')
print('=' * 45)
print(f"P(treatment > control): {result['prob_treatment_better']:.3f} ({result['prob_treatment_better']*100:.1f}%)")
print(f"Expected lift:          {result['lift_mean']:+.1%}")
print(f"95% Credible interval:  [{result['lift_ci_95'][0]:.1%}, {result['lift_ci_95'][1]:.1%}]")
print(f"Expected loss (if ship):{result['expected_loss']:.5f}")
print()

# Decision framework
prob = result['prob_treatment_better']
if prob > 0.95:
    decision = '🚀 SHIP IT (>95% probability treatment is better)'
elif prob > 0.80:
    decision = '🤔 COLLECT MORE DATA (80-95% — not conclusive enough)'
else:
    decision = '❌ KEEP CONTROL (<80% probability of improvement)'
print(f'Decision: {decision}')

In [ ]:
# Visualize Bayesian results
samples_c, samples_t = result['samples']
(alpha_c, beta_c), (alpha_t, beta_t) = result['posteriors']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Posterior distributions ──────────────────────────
ax = axes[0]
x = np.linspace(0.02, 0.10, 1000)

pdf_c = stats.beta.pdf(x, alpha_c, beta_c)
pdf_t = stats.beta.pdf(x, alpha_t, beta_t)

ax.plot(x, pdf_c, 'b-', lw=2, label=f'Control (red button)')
ax.plot(x, pdf_t, 'g-', lw=2, label=f'Treatment (green button)')
ax.fill_between(x, pdf_c, alpha=0.2, color='blue')
ax.fill_between(x, pdf_t, alpha=0.2, color='green')

ax.axvline(obs_p_control, color='blue', linestyle='--', alpha=0.7,
           label=f'Control observed: {obs_p_control:.4f}')
ax.axvline(obs_p_treatment, color='green', linestyle='--', alpha=0.7,
           label=f'Treatment observed: {obs_p_treatment:.4f}')

ax.set_xlabel('Conversion Rate')
ax.set_ylabel('Probability Density')
ax.set_title(f'Posterior Distributions\nP(treatment > control) = {result["prob_treatment_better"]:.1%}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Plot 2: Lift distribution ─────────────────────────────────
ax = axes[1]
lift_samples = (samples_t - samples_c) / samples_c * 100

ax.hist(lift_samples, bins=80, density=True, alpha=0.7, color='purple', edgecolor='none')
ax.axvline(0, color='red', lw=2, linestyle='--', label='No lift')
ax.axvline(result['lift_mean']*100, color='black', lw=2,
           label=f'Mean lift: {result["lift_mean"]*100:.1f}%')

ci = result['lift_ci_95']
ax.axvline(ci[0]*100, color='gray', lw=1.5, linestyle=':')
ax.axvline(ci[1]*100, color='gray', lw=1.5, linestyle=':',
           label=f'95% CI: [{ci[0]:.1%}, {ci[1]:.1%}]')

# Shade the 'better' region
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1],
                  0, max(lift_samples), alpha=0.1, color='green',
                  label=f'P(lift > 0%) = {result["prob_treatment_better"]:.1%}')

ax.set_xlabel('Relative Lift (%)')
ax.set_ylabel('Density')
ax.set_title('Lift Distribution (Monte Carlo)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Bayesian A/B Test — Green vs Red Checkout Button', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 3: Business Impact Analysis

In [ ]:
# Calculate expected business impact
monthly_visitors = 150_000
avg_order_value = 85  # USD

monthly_orders_control = monthly_visitors * obs_p_control
monthly_orders_treatment = monthly_visitors * obs_p_treatment
incremental_orders = monthly_orders_treatment - monthly_orders_control
incremental_revenue = incremental_orders * avg_order_value

print('BUSINESS IMPACT ANALYSIS')
print('=' * 45)
print(f'Monthly visitors:           {monthly_visitors:>10,}')
print(f'Avg order value:            {avg_order_value:>10,} USD')
print()
print(f'Control monthly orders:     {monthly_orders_control:>10,.0f}')
print(f'Treatment monthly orders:   {monthly_orders_treatment:>10,.0f}')
print(f'Incremental orders/month:   {incremental_orders:>10,.0f}')
print(f'Incremental revenue/month:  {incremental_revenue:>10,.0f} USD')
print(f'Incremental revenue/year:   {incremental_revenue*12:>10,.0f} USD')
print()

# Uncertainty bounds using 95% CI
lift_low, lift_high = result['lift_ci_95']
rev_low = monthly_visitors * obs_p_control * lift_low * avg_order_value * 12
rev_high = monthly_visitors * obs_p_control * lift_high * avg_order_value * 12
print(f'95% CI annual revenue impact: [${rev_low:,.0f}, ${rev_high:,.0f}]')

## Part 4: Common A/B Testing Mistakes

In [ ]:
print('''
COMMON A/B TESTING MISTAKES
══════════════════════════════════════════════════════════

1. PEEKING PROBLEM (Most Common!)
   ❌ Wrong: Check p-value daily, stop when p < 0.05
   ✅ Right: Decide sample size before starting, stick to it
   Why: At α=0.05, if you check 20 times, expect 1 false positive!

2. MULTIPLE TESTING
   ❌ Wrong: Test 10 variants, report the winner (p < 0.05)
   ✅ Right: Bonferroni correction (α/n) or pre-register hypothesis
   Why: With 10 tests at α=0.05, ~50% chance of at least one false positive

3. INSUFFICIENT STATISTICAL POWER
   ❌ Wrong: Run for 3 days regardless of sample size
   ✅ Right: Calculate n before experiment, run until sample achieved
   Why: Underpowered studies have high false negative rates

4. NOVELTY EFFECT
   ❌ Wrong: Conclude green button is better based on first week
   ✅ Right: Run for at least 2 business cycles (2 weeks minimum)
   Why: Users behave differently when they see something new

5. SURVIVORSHIP BIAS
   ❌ Wrong: Only analyze users who completed checkout
   ✅ Right: Intent-to-treat: analyze ALL users who saw the variant
   Why: Completion rate itself is part of what you're measuring

6. IGNORING PRACTICAL SIGNIFICANCE
   ❌ Wrong: p < 0.05, ship it!
   ✅ Right: 0.1% lift is statistically significant at n=10M but worthless
   Why: Statistical significance ≠ business significance
''')

## Challenge

1. **Power curve:** Plot sample size vs detectable effect for α=0.05, power=0.80. At what effect size does the required sample size exceed 100,000 users?

2. **Sequential testing:** Simulate the peeking problem. Run a "no effect" experiment (both variants identical, p=0.05). Check p-value after every 100 users. How often do you falsely reject H₀?

3. **Multi-variate test:** You want to test 3 button colors (red, green, blue). What sample size do you need? What correction do you apply?

4. **Real data:** Use the Titanic dataset to test: "Did women have significantly different survival rates in 1st class vs 3rd class?" State all 5 steps.